# 05 — Data Quality Audit

**Objetivo:** consolidar riesgos de calidad sin limpiar ni eliminar observaciones automáticamente.

Este notebook es un **laboratorio reproducible**. La lógica reutilizable vive en `src/argos/`;
aquí solo se parametriza, ejecuta, inspecciona evidencia y documentan decisiones.

## Principio
`missing`, `outlier` y `duplicate` son diagnósticos; no son instrucciones de `drop()`.

La salida es un **issue register** priorizado. Las transformaciones correctivas pertenecerán a Silver y deberán ser reproducibles.

In [1]:
from pathlib import Path
import tomllib
import pandas as pd

from argos.common.paths import ProjectPaths
from argos.config.registry import load_registry, discover_dataset_files

paths = ProjectPaths.discover()
paths.ensure_runtime_dirs()
DATASETS, RELATIONSHIPS = load_registry(paths.configs / "datasets.toml")
FILES = discover_dataset_files(paths.bronze, DATASETS)

with (paths.configs / "project.toml").open("rb") as handle:
    PROJECT_CONFIG = tomllib.load(handle)

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 120)

{k: p.name for k, p in FILES.items()}

{'processes': 'SECOP_II_-_Procesos_de_Contratación_20260921.csv',
 'contracts': 'SECOP_II_-_Contratos_Electrónicos_20260921.csv',
 'bidders': 'Proponentes_por_Proceso_SECOP_II_20260921.csv',
 'suppliers': 'SECOP_II_-_Proveedores_Registrados_20260921.csv',
 'additions': 'SECOP_II_-_Adiciones_20260921.csv',
 'sanctions': 'SECOPII_-_Multas_y_Sanciones_20260921.csv',
 'payments': 'SECOP_II_-_Plan_de_pagos_20260921.csv'}

In [2]:
from argos.quality.audit import audit_money_columns, summarize_quality
from argos.common.reports import write_report

cfg = PROJECT_CONFIG["profiling"]

# Reutiliza resultados previos para evitar escaneos redundantes.
column_profile = pd.read_parquet(paths.reports / "profiling" / "column_profile.parquet")
temporal_profile = pd.read_parquet(paths.reports / "profiling" / "temporal_profile.parquet")
key_analysis = pd.read_parquet(paths.reports / "profiling" / "key_analysis.parquet")
join_analysis = pd.read_parquet(paths.reports / "profiling" / "join_analysis.parquet")

money_frames = [
    audit_money_columns(
        dataset,
        path,
        sample_size=cfg["csv_sample_size"],
        temp_directory=paths.root / ".cache" / "duckdb",
    )
    for dataset, path in FILES.items()
]
money_profile = pd.concat([df for df in money_frames if not df.empty], ignore_index=True)
write_report(money_profile, paths.reports / "quality" / "money_profile.parquet")

quality_issues = summarize_quality(
    column_profile,
    temporal_profile,
    key_analysis,
    join_analysis,
    money_profile,
)
write_report(quality_issues, paths.reports / "quality" / "quality_issue_register.parquet")

quality_issues.sort_values(["severity", "dataset", "scope"]).head(200)

FileNotFoundError: [Errno 2] No such file or directory: '/Users/macm1pro/Desktop/projects/argos/argos-code/reports/profiling/temporal_profile.parquet'

In [ ]:
# Resumen ejecutivo de problemas detectados.
quality_issues.groupby(["severity", "rule"], dropna=False).size().rename("issues").reset_index()

## Criterio de salida
Cada issue material debe terminar con una decisión: `accept`, `fix_in_silver`, `exclude`, `monitor` o `needs_domain_research`.
No modifiques Bronze.

## M1 scope note

`payments` and `sanctions` remain profiled and their issues remain visible in the quality register. For the first `y_addition_180d` milestone they are explicitly **deferred**, so their unresolved defects do not silently contaminate features and do not block the addition-risk Silver gate.
